# padding-amount-formula-convT — ex2: inverse-solve `padding` from a target ConvT output size

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `padding-amount-formula-convT`. Running the final beacon cell reports progress against the `CNN: ConvT padding amount formula` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT padding amount formula` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`padding-amount-formula-convT`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "padding-amount-formula-convT"
DD_SUBTOPIC = "CNN: ConvT padding amount formula"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Inverse-solving `padding` from a desired ConvT output size — quick refresher

ConvTranspose2d output shape (no `output_padding`, dilation 1):

```
H_out = (H_in - 1) * S - 2 * P + K
```

Inverting algebraically for `P` given a desired `H_out`:

```
2 * P = (H_in - 1) * S + K - H_out
P     = ((H_in - 1) * S + K - H_out) // 2
```

**Validity.** The numerator must be `>= 0` AND even — otherwise no non-negative integer `P` produces exactly `H_out`. Return `None` when the target isn't achievable.

**Exemplar.** `H_in = 4, K = 3, S = 1`. Default `P=0` gives `H_out = 6`. To force `H_out = 4`, solve `P = ((4-1)*1 + 3 - 4) // 2 = 1`. Plug back: `(4-1)*1 - 2*1 + 3 = 4`. Correct.

### Exercise 2 — inverse-solve `padding` from a target ConvT output size

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the inverted ConvT shape formula `P = ((H_in - 1) * S + K - H_out) // 2` (with validity check) to recover the `padding` arg needed to hit a target output size, and verify against `nn.ConvTranspose2d`.
> Keywords: ConvTranspose2d, padding, inverse, decoder-design
> ```

**KCs targeted:** `convT-padding-subtracts-from-output`, `convT-padding-inverse-solve`

Implement `ex2_convT_padding_for(h_in, k, s, h_out_target)`. Given `H_in, K, S`, and a desired output length `h_out_target`, return the non-negative integer `P` that makes `nn.ConvTranspose2d` produce exactly `h_out_target`. Return `None` if no non-negative integer `P` achieves the target.

**Algebra.** Start from `H_out = (H_in - 1) * S - 2 * P + K`. Solve for `P`:
```
2 * P = (H_in - 1) * S + K - H_out
P     = ((H_in - 1) * S + K - H_out) // 2
```

**Validity checks (return `None` if any fails).**
1. `(H_in - 1) * S + K - H_out` must be `>= 0` — a negative value would mean negative padding (not a thing).
2. The same expression must be EVEN — `H_out` is reached by subtracting `2 * P` (an even number) from `(H_in - 1) * S + K`, so the gap must be even.

**Sanity check.** Plug the returned `P` back into the forward formula and confirm it hits `h_out_target`. The test does this against the actual `nn.ConvTranspose2d` module too.

**Use case.** Decoder design — you've decided the spatial size at each level (e.g. 8 → 16 → 32 → 64) and need to back-out the padding arg for each transposed conv. This is the inverted form of ex1.

In [ ]:
def ex2_convT_padding_for(h_in: int, k: int, s: int, h_out_target: int):
    """Return the integer padding P such that nn.ConvTranspose2d hits h_out_target, or None."""
    raise NotImplementedError()


def _test_ex2():
    from torch import nn

    # Direct algebra checks.
    # h_in=4, k=3, s=1, target=4 → (4-1)*1 + 3 - 4 = 2, /2 = 1 → P=1.
    assert ex2_convT_padding_for(4, 3, 1, 4) == 1
    # h_in=4, k=3, s=1, target=6 → (4-1)*1 + 3 - 6 = 0, /2 = 0 → P=0 (no padding).
    assert ex2_convT_padding_for(4, 3, 1, 6) == 0
    # h_in=4, k=3, s=1, target=2 → (4-1)*1 + 3 - 2 = 4, /2 = 2 → P=2.
    assert ex2_convT_padding_for(4, 3, 1, 2) == 2
    # h_in=8, k=4, s=2, target=16 → (8-1)*2 + 4 - 16 = 18 - 16 = 2, /2 = 1 → P=1 (canonical 2x).
    assert ex2_convT_padding_for(8, 4, 2, 16) == 1
    # h_in=5, k=3, s=2, target=9 → (5-1)*2 + 3 - 9 = 8+3-9 = 2, /2 = 1 → P=1.
    assert ex2_convT_padding_for(5, 3, 2, 9) == 1

    # Invalid: target too LARGE (would need negative padding).
    # h_in=4, k=3, s=1, target=99 → 99 > 6 (the no-pad max). Should return None.
    assert ex2_convT_padding_for(4, 3, 1, 99) is None, 'target > no-pad max must return None'
    # h_in=4, k=3, s=1, target=7 → 7 > 6. None.
    assert ex2_convT_padding_for(4, 3, 1, 7) is None

    # Invalid: parity violation (gap is odd).
    # h_in=4, k=3, s=1, target=5 → (4-1)*1 + 3 - 5 = 1 (odd). No integer P. None.
    assert ex2_convT_padding_for(4, 3, 1, 5) is None, 'odd gap must return None (no integer P)'
    # h_in=4, k=4, s=1, target=4 → (4-1)*1 + 4 - 4 = 3 (odd). None.
    assert ex2_convT_padding_for(4, 4, 1, 4) is None

    # Round-trip: solved P must reproduce h_out_target via the forward formula.
    for h_in, k, s in [(4, 3, 1), (5, 3, 2), (8, 4, 2), (10, 5, 2), (16, 3, 1)]:
        for target in range(1, 20):
            P = ex2_convT_padding_for(h_in, k, s, target)
            if P is None:
                continue
            assert P >= 0, f'returned P={P} must be non-negative'
            forward = (h_in - 1) * s - 2 * P + k
            assert forward == target, (
                f'round-trip fail: h_in={h_in},k={k},s={s},target={target} → P={P} '
                f'gives h_out={forward}'
            )

    # Cross-check against actual nn.ConvTranspose2d for several cases.
    for h_in, k, s, target in [(4, 3, 1, 4), (8, 4, 2, 16), (5, 3, 2, 9), (4, 3, 1, 6)]:
        P = ex2_convT_padding_for(h_in, k, s, target)
        assert P is not None
        ct = nn.ConvTranspose2d(in_channels=1, out_channels=1, kernel_size=k, stride=s, padding=P)
        x = t.randn(1, 1, h_in, h_in)
        actual = ct(x).shape[-1]
        assert actual == target, (
            f'nn.ConvTranspose2d with P={P} produced {actual}, expected {target}'
        )

    # Edge: target=0 only happens if (h_in-1)*s + k is even — most don't match exactly.
    # h_in=3, k=2, s=1: (3-1)*1+2 = 4, target=0 → P=2. Edge of validity.
    P_edge = ex2_convT_padding_for(3, 2, 1, 0)
    assert P_edge == 2, f'edge target=0 case: P should be 2, got {P_edge}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_convT_padding_for(h_in: int, k: int, s: int, h_out_target: int):
    gap = (h_in - 1) * s + k - h_out_target
    if gap < 0:
        return None              # target exceeds no-pad maximum
    if gap % 2 != 0:
        return None              # parity violation — no integer P
    return gap // 2
```

**Why the two validity checks.** `H_out = (H_in - 1) * S + K - 2 * P`. Solving algebraically for `P` gives `gap / 2` where `gap = (H_in-1)*S + K - H_out`. Three obstructions:
- `gap < 0` ⇒ `P < 0`, which `nn.ConvTranspose2d` rejects.
- `gap` odd ⇒ `P` non-integer; PyTorch only accepts ints.
- `gap == 0` is fine (means `P = 0`, no padding).

When you build decoder networks, parity violations are the common reason a specific `(H_in, K, S, H_target)` doesn't fit — the standard fix is to use `output_padding=1` (or to adjust `K`), not to fight the formula.

**Why this composes with `output_padding`.** The full formula adds `+ OP` to `H_out`. So if you have an off-by-one parity issue (`gap == 2*P + 1`), set `OP = 1` and the remaining `gap - 1` becomes even — and `P = (gap - 1) // 2` works. The combined `ex2 + output-padding` recipe lets you hit any spatial target.

**Decoder design recipe.** For each upsample block you decide: source size, target size, kernel, stride. Run `ex2_convT_padding_for` to get `P`. If it's `None`, either bump `K` by 1, change `S`, or set `output_padding=1`. This is exactly what `torchgan` / ARENA's DCGAN decoder helper does under the hood.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()